In [ ]:
import pandas as pd
import pyranges as pr
from matplotlib_venn import venn2
import matplotlib.pyplot as plt

In [ ]:
# ---------------------------
# 参数设置
# ---------------------------
upstream = 0
bed_file = "gar.nc_igia.bed"

# ---------------------------
# Step 1: 读取 BED
# ---------------------------
genes_pr = pr.read_bed(bed_file)
genes_df = genes_pr.df[['Chromosome','Start','End','Name','Strand']].copy()

# ---------------------------
# Step 2: 提取基因ID
# ---------------------------
genes_df['GeneID'] = genes_df['Name'].str.split('.').str[0]

# ---------------------------
# Step 3: 按基因合并转录本，取最大覆盖区间
# ---------------------------
gene_ranges = genes_df.groupby('GeneID').agg({
    'Chromosome': 'first',   # 假设同基因所有转录本在同一染色体
    'Strand': 'first',       # 假设同基因所有转录本在同一链
    'Start': 'min',          # 扩展后的 Start 取最小
    'End': 'max'             # 扩展后的 End 取最大
}).reset_index()

# ---------------------------
# Step 4: 转回 PyRanges（可选）
# ---------------------------
genes_merged_pr = pr.PyRanges(gene_ranges)

# ---------------------------
# Step 6: 保存结果
# ---------------------------
genes_merged_pr.df.to_csv("gar_genes_merged_gene.bed", sep="\t", index=False)

## CAGE-Seq 数据处理

In [ ]:
# ---------------------------
# 读取 CAGE TPM 文件
# ---------------------------
df_plus = pd.read_csv("cage_tpm/Cage-plus.tpm", sep="\t", index_col=0)
df_minus = pd.read_csv("cage_tpm/Cage-minus.tpm", sep="\t", index_col=0)

df_plus_filtered = df_plus[df_plus.iloc[:, 1:].min(axis=1) > 0.1]
df_minus_filtered = df_minus[df_minus.iloc[:, 1:].min(axis=1) > 0.1]

# 添加 strand 信息（可选）
df_plus_filtered['Strand'] = '+'
df_minus_filtered['Strand'] = '-'

# 合并
df_all = pd.concat([df_plus_filtered, df_minus_filtered])

# ---------------------------
# 解析 peak ID -> chr/start/end
# 假设 ID 格式为 "22544_Chr01_1007" 或 "1_Chr01_25349"
# ---------------------------
peak_info = df_all.index.to_series().str.split("_", expand=True)

# 生成新的 DataFrame
df_cage = pd.DataFrame({
    'Chromosome': peak_info[1],
    'Start': peak_info[2].astype(int),
    'End': peak_info[2].astype(int) + 1,   # BED 风格，单个碱基用 End = Start + 1
    'Strand': df_all['Strand']
})

## Pacbio-Seq 数据处理

In [ ]:
# 读取 PacBio TSS bed 文件
pb_tss = pd.read_csv("pacbio_TSS.bed", sep="\t", header=None, names=["chrom","start","end","transcript_id","score","strand"])

# 拆分正负链
pb_plus  = pb_tss[pb_tss['strand'] == '+'].copy()
pb_minus = pb_tss[pb_tss['strand'] == '-'].copy()

# + 链：按 start 计数，保留重复 ≥2 次的
plus_counts = pb_plus['start'].value_counts()
keep_plus_start = plus_counts[plus_counts >= 2].index
pb_plus_filt = pb_plus[pb_plus['start'].isin(keep_plus_start)]

# - 链：按 end 计数，保留重复 ≥2 次的
minus_counts = pb_minus['end'].value_counts()
keep_minus_end = minus_counts[minus_counts >= 2].index
pb_minus_filt = pb_minus[pb_minus['end'].isin(keep_minus_end)]

# 合并最终结果
pb_tss_filtered = pd.concat([pb_plus_filt, pb_minus_filt], axis=0)

print("过滤前条数:", len(pb_tss))
print("过滤后条数:", len(pb_tss_filtered))

## 整合分析

In [ ]:
df_multiTIS = pd.read_csv("table.tisseq_multiTIS.gar.csv")
target_col = ['ORF_ID', 'gene_id', 'ORF_type','chrom','strand','ORF_gstart', 'ORF_gstop',"AAseq"]
df_multiTIS_simple = df_multiTIS[target_col]

target_genelist = df_multiTIS['gene_id'].unique().tolist()
print(len(target_genelist))
df_multiTIS_filtered = genes_merged_pr.df[genes_merged_pr.df['GeneID'].isin(target_genelist)].copy()

In [ ]:
# ---------------------------
# 1: 设置上下游扩展
# ---------------------------
upstream = 1000
downstream = 1000

# ---------------------------
# 2: 构建基因 PyRanges 并扩展
# ---------------------------
# 复制并计算扩展后的 Start 和 End
genes_df = df_multiTIS_filtered.copy()

def extend_by_strand(row):
    if row['Strand'] == '+':
        start_ext = max(0, row['Start'] - upstream)  
        end_ext = row['End']                          
    else:  # '-'链
        start_ext = row['Start']                      
        end_ext = row['End'] + downstream          
    return pd.Series({'Start_ext': start_ext, 'End_ext': end_ext})

genes_df[['Start_ext', 'End_ext']] = genes_df.apply(extend_by_strand, axis=1)

In [ ]:
genes_pr_ext = genes_df[['Chromosome', 'Start_ext', 'End_ext', 'Strand', 'GeneID']].copy()

# 重命名列为标准名
genes_pr_ext = genes_pr_ext.rename(columns={
    'Start_ext': 'Start',
    'End_ext': 'End',
    'GeneID': 'Name'   
})

genes_pr_ext['Start'] = genes_pr_ext['Start'].astype(int)
genes_pr_ext['End'] = genes_pr_ext['End'].astype(int)
genes_pr_ext['Chromosome'] = genes_pr_ext['Chromosome'].astype(str)
genes_pr_ext['Strand'] = genes_pr_ext['Strand'].astype(str)
genes_pr_ext['Name'] = genes_pr_ext['Name'].astype(str)

# 构建 PyRanges 对象
genes_pr_ext = pr.PyRanges(genes_pr_ext)

In [ ]:
# ---------------------------
#  CAGE TSS
# ---------------------------
df_cage_pr = pr.PyRanges(df_cage.rename(columns={
    'Start':'Start',
    'End':'End',
    'Chromosome':'Chromosome',
    'Strand':'Strand'
}))

cage_overlap = df_cage_pr.join(genes_pr_ext, how="containment").as_df()
cage_grouped = cage_overlap.groupby('Name')['Start'].apply(list).reset_index()
cage_grouped = cage_grouped.rename(columns={'Name':'GeneID','Start':'CAGE_Starts'})

In [ ]:
# ---------------------------
#  PacBio TSS
# ---------------------------
pb_pr = pr.PyRanges(pb_tss_filtered.rename(columns={
    'chrom':'Chromosome',
    'start':'Start',
    'end':'End',
    'strand':'Strand'
}))

pb_overlap = pb_pr.join(genes_pr_ext, how="containment").as_df()
# 如果 transcript_id 需要映射到 geneID
pb_overlap['GeneID'] = pb_overlap['Name']
pb_grouped = pb_overlap.groupby('GeneID')['Start'].apply(list).reset_index()
pb_grouped = pb_grouped.rename(columns={'Start':'PB_Starts'})

In [ ]:
# ---------------------------
# 合并到原基因表
# ---------------------------
df_final = df_multiTIS_filtered.merge(cage_grouped, on='GeneID', how='left')
df_final = df_final.merge(pb_grouped, on='GeneID', how='left')

### 匹配

In [ ]:
window = 1000

df_final['CAGE_Starts'] = df_final['CAGE_Starts'].apply(lambda x: x if isinstance(x, list) else [])
df_final['PB_Starts']   = df_final['PB_Starts'].apply(lambda x: x if isinstance(x, list) else [])

# 创建字典
gene2cage = dict(zip(df_final['GeneID'], df_final['CAGE_Starts']))
gene2pb   = dict(zip(df_final['GeneID'], df_final['PB_Starts']))

In [ ]:
# 窗口大小
window = 500

# 匹配函数（只取最近位点）
def match_signals_nearest(row):
    gene = row['gene_id']
    strand = row['strand']
    
    cage_list = gene2cage.get(gene, [])
    pb_list = gene2pb.get(gene, [])
    
    pos = row['ORF_gstart']
    cage_candidates = [x for x in cage_list if pos - window <= x <= pos]
    pb_candidates   = [x for x in pb_list if pos - window <= x <= pos]

    # 取最近位点，如果没有则返回 NaN
    cage_nearest = min(cage_candidates, key=lambda x: abs(x - pos)) if cage_candidates else pd.NA
    pb_nearest   = min(pb_candidates, key=lambda x: abs(x - pos)) if pb_candidates else pd.NA
    
    return pd.Series({'cage_signal': cage_nearest, 'pacbio_signal': pb_nearest})

# 应用
df_multiTIS_simple[['cage_signal', 'pacbio_signal']] = df_multiTIS_simple.apply(match_signals_nearest, axis=1)
# 保存结果
df_multiTIS_simple.to_csv("table_multiTIS_and_multiTSS.txt", sep="\t", index=False)

In [ ]:
# 1、筛选cage 信号支持的ORF
valid_ids = df_multiTIS_simple[df_multiTIS_simple["cage_signal"].notna()]["ORF_ID"].unique()
df_selected = df_multiTIS_simple[df_multiTIS_simple["ORF_ID"].isin(valid_ids)]

# 2、判断是否存在多个tss-tis 位点
# 生成组合列
df_selected['gstart_cage_combo'] = df_selected['ORF_gstart'].astype(str) + "_" + df_selected['cage_signal'].astype(str)
# 按 gene_id 统计组合唯一数
gstart_counts = df_selected.groupby('gene_id')['gstart_cage_combo'].nunique()
# 找出至少有 2 个 ORF_gstart 的基因
genes_multi_gstart = gstart_counts[gstart_counts >= 2].index.tolist()
# 保留这些基因的所有行
df_target = df_selected[df_selected["gene_id"].isin(genes_multi_gstart)]

cage_gene = df_target["gene_id"].unique()
print("CAGE支持的multi-TSS导致的multi-TIS基因数：", len(cage_gene))

df_target.to_csv("table.multiTIS_support_by_Cage.txt", sep="\t", index=False)

In [ ]:
# 分析pcabio 支持的multi-TSS 和multi-TIS
# 1、筛选cage 信号支持的ORF
valid_ids = df_multiTIS_simple[df_multiTIS_simple["pacbio_signal"].notna()]["ORF_ID"].unique()
df_selected = df_multiTIS_simple[df_multiTIS_simple["ORF_ID"].isin(valid_ids)]

# 2、判断是否存在多个tss-tis 位点
# 生成组合列
df_selected['gstart_pacbio_combo'] = df_selected['ORF_gstart'].astype(str) + "_" + df_selected['pacbio_signal'].astype(str)
# 按 gene_id 统计组合唯一数
gstart_counts = df_selected.groupby('gene_id')['gstart_pacbio_combo'].nunique()
# 找出至少有 2 个 ORF_gstart 的基因
genes_multi_gstart = gstart_counts[gstart_counts >= 2].index.tolist()
# 保留这些基因的所有行
df_target = df_selected[df_selected["gene_id"].isin(genes_multi_gstart)]

pacbio_gene = df_target["gene_id"].unique()
print("Pacbio支持的multi-TSS导致的multi-TIS基因数：", len(pacbio_gene))

df_target.to_csv("table.multiTIS_support_by_Pacbio.txt", sep="\t", index=False)

In [ ]:
window = 1000

df_final['CAGE_Starts'] = df_final['CAGE_Starts'].apply(lambda x: x if isinstance(x, list) else [])
df_final['PB_Starts']   = df_final['PB_Starts'].apply(lambda x: x if isinstance(x, list) else [])

# 创建字典
gene2cage = dict(zip(df_final['GeneID'], df_final['CAGE_Starts']))
gene2pb   = dict(zip(df_final['GeneID'], df_final['PB_Starts']))

#### 将cage和pacbio结果合并

In [ ]:
# 转成 set
set_cage = set(cage_gene)
set_pb = set(pacbio_gene)
set_total_multiTIS = set(df_multiTIS_simple['gene_id'].tolist())

# 合并 CAGE + PacBio
set_supported = set_cage | set_pb

# 画二集合韦恩图
plt.figure(figsize=(4,4))

venn2(
    [set_supported, set_total_multiTIS],
    set_labels=('CAGE/PacBio-supported', 'All multi-TIS')
)

plt.title("Venn diagram of supported multi-TIS genes")
plt.tight_layout()

# 保存
plt.savefig( "plot_multiTIS_supported_venn.2.pdf", format='pdf', bbox_inches='tight')

plt.show()